# 4b · Retrieve & ground ARIA's answers

**Core · Live-code · ~20 min**

Use the vector store to answer questions **from the manuals**, with citations.

> Solution: [`solution/04b_retrieve_ground.ipynb`](solution/04b_retrieve_ground.ipynb).

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))
from shared.rag import RagIndex
from shared.llm import chat

# Build the index (chunks + embeds all manuals). Takes a few seconds.
index = RagIndex.from_manuals()
print(f"Indexed {len(index.chunks)} chunks")

## Step 1 — retrieve

What passages does the store return for our emergency question?

In [ ]:
for hit in index.retrieve("What do I do if O2 drops in Module B?", k=3):
    print(f"--- {hit['source']} ---")
    print(hit["text"][:200], "...\n")

## Step 2 — ground the answer

Now build a prompt that includes those passages and instructs ARIA to answer **only** from
them, citing the source. Try writing it yourself before using the helper.

In [ ]:
def grounded_answer(query, k=3):
    hits = index.retrieve(query, k=k)
    context = "\n\n---\n\n".join(f"[{h['source']}]\n{h['text']}" for h in hits)
    # TODO: write a system prompt: answer ONLY from the excerpts, cite the source file,
    #       and if the answer isn't present, say you don't have that information.
    system = "# TODO"
    return chat(f"Excerpts:\n{context}\n\nQuestion: {query}", system=system, temperature=0.2)

print(grounded_answer("What do I do if O2 drops in Module B?"))

## Step 3 — hallucination, gone

Re-ask the part-number question from Module 3. A grounded ARIA should now admit it doesn't
know, instead of inventing an answer.

In [ ]:
print(grounded_answer("What is the exact part number of the Module B scrubber cartridge?"))

# Compare with the built-in helper (shared/rag.py) used in later modules:
print("\n--- via shared.rag ---")
print(index.answer("How should we respond to a dust storm?"))

### 🌱 Stretch
See [`stretch_evaluation.ipynb`](stretch_evaluation.ipynb) to measure how good retrieval is.